# 3) The DC approximation: what market models assume about the network

## Objectives

At the end of this notebook you should be able to:

- derive the **DC power flow** from the AC equations and name each of its assumptions
- implement it — a linear system B θ = P — and check it against pandapower's `rundcpp`
- measure how far DC flows are from AC flows, and **where the difference comes from**: losses, resistance, voltage, loading
- run the same network in **PyPSA** (`pf()` and `lpf()`) and pandapower and show they agree
- explain, for the week-15 checkpoint, how the DC approximation differs from the AC power flow

Every market and planning model in this course — the PyPSA dispatch and expansion models, the nodal prices of lesson 4, flow-based market coupling in lesson 5 — represents the network with the DC approximation. It turns the non-linear power flow into a linear one, which is what makes those models linear programs. This lesson is about the price of that convenience.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pypsa

warnings.filterwarnings("ignore")
for name in ("pandapower", "pypsa", "linopy"):
    logging.getLogger(name).setLevel(logging.ERROR)

### From AC to DC

The active power sent from bus i to bus k over a branch with series impedance r + jx is, in per unit,

P_ik = |V_i|² g − |V_i||V_k| ( g cos(θ_i − θ_k) + b sin(θ_i − θ_k) ),  with g + jb = 1 / (r + jx)

Four assumptions, each reasonable on a transmission network in normal operation, turn this into a linear equation:

1. **voltages at 1 p.u.** — |V_i| = |V_k| = 1
2. **small angle differences** — sin(θ_i − θ_k) ≈ θ_i − θ_k, cos ≈ 1
3. **resistance much smaller than reactance** — r ≪ x, so g ≈ 0 and b ≈ −1/x
4. **reactive power ignored** — and with it the line charging and shunts

What remains is **P_ik = (θ_i − θ_k) / x_ik**: power flows down angle differences like current down a voltage difference. Summing over the branches at every bus gives a linear system **B θ = P** with B built from the reactances alone — closely related to lesson 2's B′, which in its XB form additionally sets transformer tap ratios to 1. One bus — the slack — has its angle fixed at zero and balances the system. With r = 0 there are no losses: the slack supplies exactly load minus generation.

### Your own DC power flow, against pandapower

In [2]:
def dc_power_flow(net):
    """DC power flow from pandapower's branch data. Returns bus angles (degrees, pandapower order) and branch flows (MW)."""
    pp.rundcpp(net)                                   # only to build the internal branch table; results are recomputed below
    ppc, order = net._ppc, net._pd2ppc_lookups["bus"]
    br = np.real(ppc["branch"])
    f, t = br[:, 0].astype(int), br[:, 1].astype(int)
    tap = np.where(br[:, 8] == 0, 1.0, br[:, 8])
    b = 1.0 / (br[:, 3] * tap)                         # series susceptance per branch, taps included
    n = ppc["bus"].shape[0]
    B = np.zeros((n, n))
    np.add.at(B, (f, f), b); np.add.at(B, (t, t), b)
    np.add.at(B, (f, t), -b); np.add.at(B, (t, f), -b)
    P = np.zeros(n)                                   # injections: generation minus load (MW)
    np.add.at(P, order[net.gen.bus.to_numpy()], net.gen.p_mw.to_numpy())
    np.add.at(P, order[net.load.bus.to_numpy()], -net.load.p_mw.to_numpy())
    slack = order[net.ext_grid.bus.to_numpy()][0]
    keep = np.array([i for i in range(n) if i != slack])
    theta = np.zeros(n)
    theta[keep] = np.linalg.solve(B[np.ix_(keep, keep)], P[keep] / ppc["baseMVA"])
    flows = b * (theta[f] - theta[t]) * ppc["baseMVA"]
    return np.degrees(theta)[order[net.bus.index]], flows


net = pn.case14()
angles, flows = dc_power_flow(net)
pp.rundcpp(net)
reference = np.r_[net.res_line.p_from_mw.to_numpy(), net.res_trafo.p_hv_mw.to_numpy()]
assert np.abs(flows - reference).max() < 1e-9 and np.abs(angles - net.res_bus.va_degree.to_numpy()).max() < 1e-9
print(f"own DC power flow = pandapower rundcpp: largest difference {np.abs(flows - reference).max():.1e} MW")

own DC power flow = pandapower rundcpp: largest difference 2.0e-13 MW


### DC against AC on the IEEE 14-bus system

In [3]:
def ac_and_dc(net):
    """Branch flows (MW, lines then transformers), angles and slack output of the AC and the DC power flow."""
    out = {}
    pp.runpp(net, numba=False)
    out["ac"] = np.r_[net.res_line.p_from_mw.to_numpy(), net.res_trafo.p_hv_mw.to_numpy()]
    out["ac_angle"], out["ac_slack"] = net.res_bus.va_degree.to_numpy().copy(), net.res_ext_grid.p_mw.iloc[0]
    out["vm_range"] = (net.res_bus.vm_pu.min(), net.res_bus.vm_pu.max())
    pp.rundcpp(net)
    out["dc"] = np.r_[net.res_line.p_from_mw.to_numpy(), net.res_trafo.p_hv_mw.to_numpy()]
    out["dc_angle"], out["dc_slack"] = net.res_bus.va_degree.to_numpy().copy(), net.res_ext_grid.p_mw.iloc[0]
    return out


r = ac_and_dc(pn.case14())
err = r["dc"] - r["ac"]
print(f"branch flows: largest AC flow {np.abs(r['ac']).max():.1f} MW; DC error mean {np.abs(err).mean():.2f} MW, "
      f"largest {np.abs(err).max():.2f} MW; angles differ by up to {np.abs(r['dc_angle'] - r['ac_angle']).max():.2f} degrees")
print(f"slack: AC {r['ac_slack']:.1f} MW, DC {r['dc_slack']:.1f} MW -> difference {r['ac_slack'] - r['dc_slack']:.1f} MW (the losses)")
fig, ax = plt.subplots(figsize=(5, 4.6))
ax.scatter(r["ac"], r["dc"], s=18)
lim = 1.1 * np.abs(r["ac"]).max()
ax.plot([-lim, lim], [-lim, lim], color="grey", lw=0.8, ls="--")
ax.set(xlabel="AC flow (MW)", ylabel="DC flow (MW)", title="IEEE 14-bus: every branch, DC vs AC", xlim=(-lim, lim), ylim=(-lim, lim))
plt.tight_layout()
plt.show()

branch flows: largest AC flow 156.9 MW; DC error mean 1.25 MW, largest 9.04 MW; angles differ by up to 1.15 degrees
slack: AC 232.4 MW, DC 219.0 MW -> difference 13.4 MW (the losses)


On the IEEE 14-bus system the DC flows follow the AC flows closely — the mean error is 1.25 MW — but not exactly. The largest error, 9 MW, sits on the largest flow, the line carrying 157 MW out of the slack bus, which DC puts at 148 MW. The reason is in the second line of output: the AC slack supplies 232.4 MW, the DC slack 219.0 MW. The difference, **13.4 MW, is the network's losses**, which DC cannot represent because it has set every resistance to zero. Those losses are supplied by the slack and flow out over the lines next to it, so that is where DC underestimates most. The angles differ by up to 1.15° — DC is a good sketch of where power flows, not an exact account of it.

### When does the approximation degrade?

In [4]:
def errors(load_scale=1.0, r_scale=1.0):
    net = pn.case14()
    net.load[["p_mw", "q_mvar"]] *= load_scale
    net.gen.p_mw *= load_scale
    net.line.r_ohm_per_km *= r_scale
    r = ac_and_dc(net)
    e = np.abs(r["dc"] - r["ac"])
    return {"largest_flow_MW": np.abs(r["ac"]).max(), "mean_error_MW": e.mean(), "largest_error_MW": e.max(),
            "largest_error_%_of_flow": 100 * e.max() / np.abs(r["ac"]).max(), "losses_MW": r["ac_slack"] - r["dc_slack"],
            "lowest_vm_pu": r["vm_range"][0]}


loading = pd.DataFrame({s: errors(load_scale=s) for s in [0.5, 1.0, 1.5, 2.0, 2.5]}).T.rename_axis("load scale")
resistance = pd.DataFrame({s: errors(r_scale=s) for s in [1, 2, 4]}).T.rename_axis("line resistance x")
print("growing load (generation scaled with it):")
print(loading.round(2).to_string())
print("\ngrowing line resistance:")
print(resistance.round(2).to_string())

growing load (generation scaled with it):
            largest_flow_MW  mean_error_MW  largest_error_MW  largest_error_%_of_flow  losses_MW  lowest_vm_pu
load scale                                                                                                    
0.5                   75.95           0.40              2.03                     2.67       3.31          1.01
1.0                  156.88           1.25              9.04                     5.76      13.39          1.01
1.5                  244.87           3.31             23.11                     9.44      32.13          1.00
2.0                  341.84           6.41             46.16                    13.50      61.46          0.97
2.5                  450.98          10.90             81.39                    18.05     104.59          0.94

growing line resistance:
                   largest_flow_MW  mean_error_MW  largest_error_MW  largest_error_%_of_flow  losses_MW  lowest_vm_pu
line resistance x                    

The two tables show where the assumptions break. **With load**, the error grows faster than the flows: the largest error is 2.7 % of the largest flow at half load, 5.8 % at base load and 18 % at two and a half times the load. Losses grow with the square of the flows (3.3 MW at half load, 13.4 MW at base load — four times as much for twice the flow), and at high load the voltages sag to 0.94 p.u., breaking the first assumption as well. **With resistance**, DC fails quickly: doubling every line's resistance doubles the relative error to 12 %; at four times the resistance the largest error reaches 45 % of the largest flow and losses 175 MW, none of which DC sees. That is the same weakness as lesson 2's fast decoupled method, for the same reason: both build on the reactances alone. DC power flow is a transmission-network tool for normal operation — it degrades exactly in the stressed situations a security study cares about.

### PyPSA and pandapower on one network

PyPSA's `pf()` solves the AC power flow, `lpf()` the DC one — the same network representation its optimisation uses. To compare it with pandapower we build the same network in both tools from one data table: a **stylised Croatian 400 kV network** of five nodes, with typical 400 kV overhead-line parameters (0.028 Ω/km, 0.32 Ω/km, 11.5 nF/km) and illustrative lengths, loads and generation — not the real grid's data. (PyPSA can also import pandapower networks with `import_from_pandapower_net`; for the IEEE 14-bus case in the tested versions that import did not reproduce the network — flows came out as NaN — so we build natively, which also tests the solvers rather than a converter.)

In [5]:
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
R_KM, X_KM, C_KM = 0.028, 0.32, 11.5                      # ohm/km, ohm/km, nF/km
LOAD = {"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250}
GEN = {"Rijeka": 500, "Split": 600, "Velebit": 300}         # MW; the slack sits at Zagreb


def croatia_pandapower():
    net = pp.create_empty_network(sn_mva=100)
    bus = {name: pp.create_bus(net, 400, name=name) for name in BUSES}
    pp.create_ext_grid(net, bus["Zagreb"], vm_pu=1.0)
    for name, p in GEN.items():
        pp.create_gen(net, bus[name], p_mw=p, vm_pu=1.0)
    for name, p in LOAD.items():
        pp.create_load(net, bus[name], p_mw=p, q_mvar=0.2 * p)
    for _, line in LINES.iterrows():
        pp.create_line_from_parameters(net, bus[line.bus0], bus[line.bus1], line.km, R_KM, X_KM, C_KM, max_i_ka=2.0)
    return net


def croatia_pypsa():
    n = pypsa.Network()
    n.add("Bus", BUSES, v_nom=400)
    n.add("Generator", "slack", bus="Zagreb", control="Slack")
    for name, p in GEN.items():
        n.add("Generator", f"{name} gen", bus=name, p_set=p, control="PV")
    for name, p in LOAD.items():
        n.add("Load", f"{name} load", bus=name, p_set=p, q_set=0.2 * p)
    for _, line in LINES.iterrows():
        n.add("Line", f"{line.bus0}-{line.bus1}", bus0=line.bus0, bus1=line.bus1, r=R_KM * line.km, x=X_KM * line.km,
              b=2 * np.pi * 50 * C_KM * 1e-9 * line.km, s_nom=np.sqrt(3) * 400 * 2.0)
    return n


net = croatia_pandapower()
pp.runpp(net, numba=False)
ac_pp = net.res_line.p_from_mw.to_numpy().copy()          # copy: rundcpp overwrites the results table
losses = net.res_line.pl_mw.sum()
vm_ac = net.res_bus.vm_pu.to_numpy().copy()
pp.rundcpp(net)
dc_pp = net.res_line.p_from_mw.to_numpy().copy()
n_ac, n_dc = croatia_pypsa(), croatia_pypsa()
n_ac.pf()
n_dc.lpf()
assert np.abs(n_ac.lines_t.p0.iloc[0].to_numpy() - ac_pp).max() < 1e-6
assert np.abs(n_dc.lines_t.p0.iloc[0].to_numpy() - dc_pp).max() < 1e-6
print("PyPSA pf() = pandapower runpp and PyPSA lpf() = pandapower rundcpp, to 1e-6 MW")
print(f"AC losses {losses:.1f} MW; voltages {vm_ac.min():.3f}-{vm_ac.max():.3f} p.u. in the AC solution")
pd.DataFrame({"AC (both tools)": ac_pp, "DC (both tools)": dc_pp, "DC - AC": dc_pp - ac_pp}, index=n_ac.lines.index).round(1)

PyPSA pf() = pandapower runpp and PyPSA lpf() = pandapower rundcpp, to 1e-6 MW
AC losses 7.0 MW; voltages 1.000-1.026 p.u. in the AC solution


                AC (both tools)  DC (both tools)  DC - AC
name                                                     
Zagreb-Rijeka            -348.4           -352.7     -4.3
Rijeka-Velebit           -151.2           -152.7     -1.5
Velebit-Split             148.3            147.3     -1.0
Zagreb-Split             -119.8           -122.1     -2.3
Zagreb-Osijek              75.2             74.9     -0.4
Osijek-Split             -175.2           -175.1      0.0

PyPSA's `pf()` and pandapower's `runpp` give the same AC flows, and `lpf()` and `rundcpp` the same DC flows, so the two tools agree completely when given the same network. On this 400 kV network DC is much closer to AC than on the IEEE 14-bus system: the largest error is 4.3 MW on the 348 MW Zagreb–Rijeka flow (1.2 %). The assumptions fit: a 400 kV line's resistance is only about 0.09 of its reactance (0.028 against 0.32 Ω/km), the voltages stay between 1.000 and 1.026 p.u., and losses are 7 MW on 1,900 MW of load — 0.4 %. This is why DC flows are an accepted approximation in market and planning models of high-voltage transmission, and why the same models should not be trusted for distribution grids or heavily stressed networks.

One practical lesson from building this comparison: pandapower writes its results into the same tables every time it solves. A result saved as `net.res_line.p_from_mw.to_numpy()` can be a view of that table and change silently when the next power flow runs — always take a `.copy()` before solving again.

## Check your understanding

**The approximation**

1. Which of the four assumptions removes the losses, and which removes reactive power? What happens to the slack bus's output as a result?
2. Why does the DC approximation fit a 400 kV network better than the IEEE 14-bus system?

**Using it**

1. A market model uses DC flows to check line limits. Name one situation in which a line looks within its limit in the DC model but is overloaded in reality.
2. For the week-15 checkpoint: in three sentences, how does the DC power flow differ from the AC power flow, and when does the difference matter?